In [1]:
import numpy as np
import pandas as pd

In [ ]:
# Load the SMS Spam Collection dataset into a pandas DataFrame
# We must specify encoding='latin-1' to handle the non-standard characters present in this specific text file
df = pd.read_csv('../data/spam.csv', encoding='latin-1')

# Display the raw dataframe to see its initial structure and the extra unnamed columns
df

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


In [5]:
# Isolate the target (v1) and the feature (v2) columns, dropping the unnamed NaN columns
df = df[['v1', 'v2']]

# Rename the columns to make them intuitive and highly readable
df.columns = ['label', 'message']

# Display the first 5 rows of our cleaned DataFrame structure
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
print(df.isnull().sum())
print(df.duplicated().sum())

label      0
message    0
dtype: int64
403


In [7]:
# Drop the duplicate rows
df = df.drop_duplicates()

# Verify the new shape of the dataset to confirm duplicates were removed
print("--- New Dataset Shape ---")
print(df.shape)

# Inspect the distribution of our target labels to check for class imbalance
print("\n--- Class Distribution ---")
print(df['label'].value_counts())

--- New Dataset Shape ---
(5169, 2)

--- Class Distribution ---
label
ham     4516
spam     653
Name: count, dtype: int64


In [8]:
# Convert all characters in the 'message' column to lowercase
df['message'] = df['message'].str.lower()

# Display the first 5 rows to verify all text is now lowercase
df.head()

,label,message
0,ham,"go until jurong point, crazy.. available only ..."
1,ham,ok lar... joking wif u oni...
2,spam,free entry in 2 a wkly comp to win fa cup fina...
3,ham,u dun say so early hor... u c already then say...
4,ham,"nah i don't think he goes to usf, he lives aro..."


In [9]:
#Remove Punctuation

import re

# Use regular expressions to remove all punctuation
# r'[^\w\s]' matches anything that is NOT an alphanumeric character (\w) or whitespace (\s)
df['message'] = df['message'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

In [13]:
df["message"].isnull().sum()

np.int64(0)

In [ ]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Yernar\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [15]:
from nltk.tokenize import word_tokenize

# Apply the word_tokenize function to split each message into a list of individual words
df['tokens'] = df['message'].apply(word_tokenize)

# Display the original message alongside the new list of tokens to see the transformation
print(df[['message', 'tokens']].head())

                                             message  \
0  go until jurong point crazy available only in ...   
1                            ok lar joking wif u oni   
2  free entry in 2 a wkly comp to win fa cup fina...   
3        u dun say so early hor u c already then say   
4  nah i dont think he goes to usf he lives aroun...   

                                              tokens  
0  [go, until, jurong, point, crazy, available, o...  
1                     [ok, lar, joking, wif, u, oni]  
2  [free, entry, in, 2, a, wkly, comp, to, win, f...  
3  [u, dun, say, so, early, hor, u, c, already, t...  
4  [nah, i, dont, think, he, goes, to, usf, he, l...  


In [16]:
from nltk.corpus import stopwords

# Download the standard list of English stopwords from NLTK
nltk.download('stopwords')

# Store the English stopwords in a set for much faster processing
stop_words = set(stopwords.words('english'))

# Apply a filter to keep only the tokens that are NOT in our stop_words list
df['tokens'] = df['tokens'].apply(lambda x: [word for word in x if word not in stop_words])

# Display the first 5 rows to see the filtered token lists
print(df['tokens'].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Yernar\AppData\Roaming\nltk_data...


0    [go, jurong, point, crazy, available, bugis, n...
1                       [ok, lar, joking, wif, u, oni]
2    [free, entry, 2, wkly, comp, win, fa, cup, fin...
3        [u, dun, say, early, hor, u, c, already, say]
4    [nah, dont, think, goes, usf, lives, around, t...
Name: tokens, dtype: object


[nltk_data]   Unzipping corpora\stopwords.zip.


In [ ]:
# Lemmatization
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')

lemmatizer = WordNetLemmatizer() # Лемматизация работает только с токенами, а не с текстом

# Apply the stemmer to chop off prefixes/suffixes and find the root stem
df['tokens'] = df['tokens'].apply(
    lambda words: [lemmatizer.lemmatize(word) for word in words]
)

df.head()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Yernar\AppData\Roaming\nltk_data...


,label,message,tokens
0,ham,go until jurong point crazy available only in ...,"[go, jurong, point, crazy, available, bugis, n..."
1,ham,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]"
2,spam,free entry in 2 a wkly comp to win fa cup fina...,"[free, entry, 2, wkly, comp, win, fa, cup, fin..."
3,ham,u dun say so early hor u c already then say,"[u, dun, say, early, hor, u, c, already, say]"
4,ham,nah i dont think he goes to usf he lives aroun...,"[nah, dont, think, go, usf, life, around, though]"


In [ ]:
#Join tokens back into text
df['processed_message'] = df['tokens'].apply(lambda x: " ".join(x))
df[['tokens','processed_message']].head()

,tokens,processed_message
0,"[go, jurong, point, crazy, available, bugis, n...",go jurong point crazy available bugis n great ...
1,"[ok, lar, joking, wif, u, oni]",ok lar joking wif u oni
2,"[free, entry, 2, wkly, comp, win, fa, cup, fin...",free entry 2 wkly comp win fa cup final tkts 2...
3,"[u, dun, say, early, hor, u, c, already, say]",u dun say early hor u c already say
4,"[nah, dont, think, go, usf, life, around, though]",nah dont think go usf life around though


In [19]:
df.head()

,label,message,tokens,processed_message
0,ham,go until jurong point crazy available only in ...,"[go, jurong, point, crazy, available, bugis, n...",go jurong point crazy available bugis n great ...
1,ham,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]",ok lar joking wif u oni
2,spam,free entry in 2 a wkly comp to win fa cup fina...,"[free, entry, 2, wkly, comp, win, fa, cup, fin...",free entry 2 wkly comp win fa cup final tkts 2...
3,ham,u dun say so early hor u c already then say,"[u, dun, say, early, hor, u, c, already, say]",u dun say early hor u c already say
4,ham,nah i dont think he goes to usf he lives aroun...,"[nah, dont, think, go, usf, life, around, though]",nah dont think go usf life around though


In [ ]:
# --- SAVE THE CLEANED DATA ---

# Save our final dataframe to a CSV file.
# We set index=False so Pandas doesn't create a useless column of row numbers!
df.to_csv('../data/spam_cleaned.csv', index=False)

print("Cleaned data saved as 'cleaned_spam_data.csv'")

Cleaned data saved as 'cleaned_spam_data.csv'
